# 3. Fit masses and mixing

Build insertion matrices, remove monomials involving forbidden VEVs, and repeatedly fit the retained parameters.

**Execution status:** the required JSON files are absent. The original outputs are archived locally; this cleaned notebook has not been run against the missing catalogue.

## Imports and conversion helpers

In [ ]:
import json
import math
import numpy as np
from numpy import linalg as LA
from scipy.optimize import least_squares, minimize


In [ ]:
def deviation(measured, exp):
    """Return the absolute percentage deviation from a nonzero target."""
    return abs(measured - exp) / exp * 100


## Filter monomials and reduce singlet coordinates

The fit removes monomials containing a VEV selected for zeroing, keeps up to `n_o1_coef_per_entry` per entry, and drops unused coordinates. The remaining schema uses `-1` for nonperturbative singlets and `+1` for perturbative singlets.

In [ ]:
def number_of_fields(matrix_up, matrix_down, vevs_to_be_killed):
    """Remove forbidden monomials, truncate entries, and reduce singlet coordinates.

    The schema marks retained nonperturbative fields by -1 and perturbative
    fields by +1. Field counts and truncation settings come from notebook state."""
    global flag, max_n_o1_coef_per_entry, n_o1_coef_per_entry
    matrix_up_cleaned = [[[], [], []], [[], [], []], [[], [], []]]
    matrix_down_cleaned = [[[], [], []], [[], [], []], [[], [], []]]
    for i in range(3):
        for j in range(3):
            for k in range(len(matrix_up[i][j])):
                check = any((matrix_up[i][j][k][l] != 0 for l in vevs_to_be_killed))
                if check == False and len(matrix_up_cleaned[i][j]) < n_o1_coef_per_entry:
                    matrix_up_cleaned[i][j].append(matrix_up[i][j][k])
            for k in range(len(matrix_down[i][j])):
                check = any((matrix_down[i][j][k][l] != 0 for l in vevs_to_be_killed))
                if check == False and len(matrix_down_cleaned[i][j]) < n_o1_coef_per_entry:
                    matrix_down_cleaned[i][j].append(matrix_down[i][j][k])
    field_usage = np.zeros(shape=n_of_phi)
    for i in range(3):
        for j in range(3):
            if flag == 'pick all':
                if len(matrix_up_cleaned[i][j]) != 0:
                    for k in range(len(matrix_up_cleaned[i][j])):
                        field_usage += np.array(matrix_up_cleaned[i][j][k])
                if len(matrix_down_cleaned[i][j]) != 0:
                    for k in range(len(matrix_down_cleaned[i][j])):
                        field_usage += np.array(matrix_down_cleaned[i][j][k])
            elif flag == 'pick one':
                field_usage += np.array(matrix_up_cleaned[i][j]) + np.array(matrix_down_cleaned[i][j])
    for i in range(n_of_phi):
        if field_usage[i] != 0:
            if i < n_of_non_pert_phi:
                field_usage[i] = -1
            else:
                field_usage[i] = 1
    unused_indices = [index for index, value in enumerate(field_usage) if value == 0]
    cut_matrix_up = [[[], [], []], [[], [], []], [[], [], []]]
    cut_matrix_down = [[[], [], []], [[], [], []], [[], [], []]]
    for i in range(3):
        for j in range(3):
            for k in range(len(matrix_up_cleaned[i][j])):
                cut_matrix_up[i][j].append([value for ii, value in enumerate(matrix_up_cleaned[i][j][k]) if ii not in unused_indices])
            for k in range(len(matrix_down_cleaned[i][j])):
                cut_matrix_down[i][j].append([value for ii, value in enumerate(matrix_down_cleaned[i][j][k]) if ii not in unused_indices])
    return (field_usage[field_usage != 0], cut_matrix_up, cut_matrix_down)


## Build and order Yukawa insertion matrices

Models must carry both Yukawa coverage flags. Nonperturbative exponents receive a larger weight derived from the configured VEV ranges. Up-sector insertion lists are mirrored; fitted coefficients are nevertheless counted independently for each matrix entry.

In [ ]:
def create_Yukawa_matrices(lib):
    """Build ordered insertion matrices for models with both Yukawa flags."""
    global flag, max_n_o1_coef_per_entry
    up_matrices = {}
    down_matrices = {}
    for i in lib:
        if lib[i][-1] == 1 and 'up yes' in lib[i] and ('down yes' in lib[i]):
            matrix_up = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
            matrix_down = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
            insertions_list_up = [[[], [], []], [[], [], []], [[], [], []]]
            insertions_list_down = [[[], [], []], [[], [], []], [[], [], []]]
            for j in range(3, len(lib[i]) - 1):
                if not isinstance(lib[i][j], list):
                    continue
                for k in lib[i][j][2]:
                    if lib[i][j][0] == 'up':
                        insertions_list_up[k[0] - 1][k[1] - 1].append(lib[i][j][1])
                    if lib[i][j][0] == 'down':
                        insertions_list_down[k[0] - 1][k[1] - 1].append(lib[i][j][1])
            if flag == 'pick all':
                n_of_non_pert_phi = len(lib[i][0])
                up_matrix = sort_by_P(insertions_list_up, n_of_non_pert_phi)
                down_matrix = sort_by_P(insertions_list_down, n_of_non_pert_phi)
                for j in range(3):
                    for k in range(3):
                        if j <= k:
                            up_matrix[j][k] = up_matrix[j][k][:min(len(up_matrix[j][k]), max_n_o1_coef_per_entry)]
                            up_matrix[k][j] = up_matrix[j][k]
                        down_matrix[j][k] = down_matrix[j][k][:min(len(down_matrix[j][k]), max_n_o1_coef_per_entry)]
                up_matrices[i] = up_matrix
                down_matrices[i] = down_matrix
    return (up_matrices, down_matrices)


In [ ]:
def sort_by_P(insertion_matrix, n_of_Phi):
    """Order insertions by weighted degree, using the configured VEV ranges."""
    sorted_matrix = [[0, 0, 0], [0, 0, 0], [0, 0, 0]]
    non_perturbative_weight = math.ceil(np.emath.logn((pert_range[0] + pert_range[1]) * 0.5, (non_pert_range[0] + non_pert_range[1]) * 0.5))
    for i in range(3):
        for j in range(3):
            orders = []
            for k in range(len(insertion_matrix[i][j])):
                tot_order = 0
                for l in range(len(insertion_matrix[i][j][k])):
                    if l < n_of_Phi:
                        tot_order += non_perturbative_weight * insertion_matrix[i][j][k][l]
                    else:
                        tot_order += insertion_matrix[i][j][k][l]
                orders.append(tot_order)
            sorted_elements = [x for _, x in sorted(zip(orders, insertion_matrix[i][j]))]
            sorted_matrix[i][j] = sorted_elements
    return sorted_matrix


In [ ]:
def n_of_o1_coeffs(up, down):
    """Count independently fitted coefficients in the up and down sectors."""
    global flag
    up_counter = 0
    down_counter = 0
    if flag == 'pick all':
        for i in range(3):
            for j in range(3):
                for k in range(len(up[i][j])):
                    up_counter += 1
                for k in range(len(down[i][j])):
                    down_counter += 1
    elif flag == 'pick one':
        up_counter = 9
        down_counter = 9
    return (up_counter, down_counter)


## Real rotations and CKM objective

The CKM stage fits magnitudes using real orthogonal matrices. A complex CP phase is not fitted. Accepted rotations are reconstructed from the optimizer result.

In [ ]:
def generate_orthogonal_matrix(x1, x2):
    """Construct a real orthogonal frame from two parameterized directions.

    Requires nondegenerate directions and a nonzero third component of x1."""
    v1 = np.array(x1)
    v2 = np.array([x2[0], x2[1], -(x1[0] * x2[0] + x1[1] * x2[1]) / x1[2]])
    v3 = np.cross(v1, v2)
    v1 = v1 / math.sqrt(v1[0] ** 2 + v1[1] ** 2 + v1[2] ** 2)
    v2 = v2 / math.sqrt(v2[0] ** 2 + v2[1] ** 2 + v2[2] ** 2)
    v3 = v3 / math.sqrt(v3[0] ** 2 + v3[1] ** 2 + v3[2] ** 2)
    matrix = np.stack([v1, v2, v3], axis=1)
    return matrix


In [ ]:
def CKM_loss(x):
    """Return the relative squared error of the real CKM magnitudes."""
    x1 = [x[0], x[1], x[2]]
    x2 = [x[3], x[4]]
    x3 = [x[5], x[6], x[7]]
    x4 = [x[8], x[9]]
    Uu = generate_orthogonal_matrix(x1, x2)
    Ud = generate_orthogonal_matrix(x3, x4)
    CKM = np.transpose(Uu) @ Ud
    loss_CKM = 0
    for i in range(3):
        for j in range(3):
            loss_CKM += ((abs(CKM[i][j]) - exp_CKM[i][j]) / exp_CKM[i][j]) ** 2
    return loss_CKM


## Parameter bounds and angular rotations

The parameter order is retained singlet VEVs, up/down/lepton coefficients, the up-Higgs VEV, and twelve rotation angles. The down-Higgs VEV is computed as $\sqrt{174^2-H_u^2}$.

In [ ]:
def from_symbols_to_bounds(schema, tot_para, relaxed):
    """Build bounds for singlets, coefficients, the Higgs VEV, and rotations."""
    bounds = []
    n_of_Phi = len(schema)
    for i in range(tot_para):
        if i < n_of_Phi:
            if schema[i] == -1:
                bounds.append([non_pert_range[0], non_pert_range[1]])
            if schema[i] == 1:
                if relaxed == True:
                    bounds.append([pert_range[0], pert_range[1]])
                else:
                    bounds.append([pert_range[0], pert_range[1]])
        if i >= n_of_Phi and i < n_of_Phi + n_o1:
            if relaxed == True:
                bounds.append([o1_range[0], o1_range[1]])
            else:
                bounds.append([o1_range[0], o1_range[1]])
        if i == n_of_Phi + n_o1:
            bounds.append([Hu_bounds[0], Hu_bounds[1]])
        if i > n_of_Phi + n_o1:
            if relaxed == True:
                bounds.append([-np.inf, np.inf])
            else:
                bounds.append([0, 6.28])
    return bounds


In [ ]:
def generate_parametrised_orthogonal_matrix(y):
    """Construct a real rotation matrix from three angles in radians."""
    ca = np.cos(y[0])
    sa = np.sin(y[0])
    cb = np.cos(y[1])
    sb = np.sin(y[1])
    cg = np.cos(y[2])
    sg = np.sin(y[2])
    matrix = np.array([[cb * cg, sa * sb * cg - ca * sg, ca * sb * cg + sa * sg], [cb * sg, sa * sb * sg + ca * cg, ca * sb * sg - sa * cg], [-sb, sa * cb, ca * cb]])
    return matrix


## Numerical settings and targets

The random seed and maximum CKM attempts are explicit. Review VEV/coefficient ranges, monomial limits, acceptance thresholds, and targets before a scan. Target sources, scale, and units still need author confirmation.

In [ ]:
non_pert_range = [0.01, 0.07]
pert_range = [0.1, 0.7]
Hu_bounds = [40, 170]
o1_range = [-7.0, 7.0]
max_n_o1_coef_per_entry = 100
n_o1_coef_per_entry = 2
UV_range = [-5, 5]
flag = 'pick all'
threshold = 10 ** (-4)
CKM_threshold = 0.1
Mt = 172.4
Mc = 1.27
Mu = 0.00216
Mb = 4.18
Ms = 0.093
Md = 0.00467
Mtau = 1.77682
Mmu = 0.1057
Me = 0.000511
exp_CKM = [[0.97373, 0.2243, 0.00382], [0.221, 0.975, 0.0408], [0.0086, 0.0415, 1.014]]
Mu_h = np.array([[Mt, 0.0, 0.0], [0.0, Mc, 0.0], [0.0, 0.0, Mu]])
Md_h = np.array([[Mb, 0.0, 0.0], [0.0, Ms, 0.0], [0.0, 0.0, Md]])
Me_h = np.array([[Mtau, 0.0, 0.0], [0.0, Mmu, 0.0], [0.0, 0.0, Me]])
RANDOM_SEED = 42
CKM_MAX_ATTEMPTS = 100
rng = np.random.default_rng(RANDOM_SEED)


## Build and export insertion matrices

`insertions_matrices.json` supplies the original singlet-coordinate matrices used by the final scan.

In [ ]:
with open('library_ALL_v4.json', 'r') as file:
    library = json.load(file)


In [ ]:
matrices = create_Yukawa_matrices(library)
with open('insertions_matrices.json', 'w') as file:
    json.dump(matrices, file, indent=2)


## Mass-matrix residuals

The residual vector combines target-normalized quark matrices with unnormalized charged-lepton matrices. The single cost threshold weights the sectors differently. Accepted matrices are evaluated afresh at the final fitted parameters.

In [ ]:
def loss_function(x, Pup, Pdown, Uu, Ud, return_matrices=False):
    """Return fitting residuals, or mass matrices evaluated at x.

    Quark residuals are target-normalized; lepton residuals are unnormalized."""
    current_try_up = [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
    current_try_down = [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
    current_try_leptons = [[0.0, 0.0, 0.0], [0.0, 0.0, 0.0], [0.0, 0.0, 0.0]]
    Hu = x[-13]
    Hd = math.sqrt(174 ** 2 - Hu ** 2)
    counter = 0
    for i in range(3):
        for j in range(3):
            for l in range(len(Pup[i][j])):
                temp_try_up = 1
                for k in range(n_of_used_Phi):
                    if Pup[i][j][l][k] != 0:
                        temp_try_up *= x[k] ** Pup[i][j][l][k]
                temp_try_up *= x[n_of_used_Phi + counter] * Hu
                counter += 1
                current_try_up[i][j] += temp_try_up
    counter = 0
    for i in range(3):
        for j in range(3):
            for l in range(len(Pdown[i][j])):
                temp_try_down = 1
                for k in range(n_of_used_Phi):
                    if Pdown[i][j][l][k] != 0:
                        temp_try_down *= x[k] ** Pdown[i][j][l][k]
                temp_try_down *= x[n_of_used_Phi + n_o1_up + counter] * Hd
                counter += 1
                current_try_down[i][j] += temp_try_down
    counter = 0
    for i in range(3):
        for j in range(3):
            for l in range(len(Pdown[i][j])):
                temp_try_leptons = 1
                for k in range(n_of_used_Phi):
                    if Pdown[i][j][l][k] != 0:
                        temp_try_leptons *= x[k] ** Pdown[i][j][l][k]
                temp_try_leptons *= x[n_of_used_Phi + n_o1_up + n_o1_down + counter] * Hd
                counter += 1
                current_try_leptons[i][j] += temp_try_leptons
    if return_matrices:
        return tuple((np.asarray(matrix) for matrix in (current_try_up, current_try_down, current_try_leptons)))
    LHSup = LA.inv(Uu @ Mu_h) @ current_try_up
    LHSdown = LA.inv(Ud @ Md_h) @ current_try_down
    LHSleptons = current_try_leptons
    RHSup = generate_parametrised_orthogonal_matrix(x[tot_para - 3:tot_para])
    RHSdown = generate_parametrised_orthogonal_matrix(x[tot_para - 6:tot_para - 3])
    RHSleptonsU = generate_parametrised_orthogonal_matrix(x[tot_para - 9:tot_para - 6])
    RHSleptonsV = generate_parametrised_orthogonal_matrix(x[tot_para - 12:tot_para - 9])
    RHSleptons = RHSleptonsU @ Me_h @ RHSleptonsV.transpose()
    loss = list((LHSup - RHSup).flatten()) + list((LHSdown - RHSdown).flatten()) + list((LHSleptons - RHSleptons).flatten())
    return loss


## Load model preselection and single-Higgs VEV-zero choices

The construction of `good_pheno_models.json` is external to this repository.

In [ ]:
with open('good_pheno_models.json', 'r') as file:
    promising_models = json.load(file)


In [ ]:
with open('mu_term_library2.json', 'r') as file:
    mu_term = json.load(file)


In [ ]:
with open('killed_vevs_library_single_higgs_models.json') as file:
    killed_vevs_library = json.load(file)


## Run repeated fits and save accepted candidates

The default is 50 restarts per model. CKM retries are bounded; fitting errors surface normally. Result keys include restart and VEV-choice indices. `viable_models.json` stores accepted candidates and `fit_settings.json` records the main settings. These candidates have not undergone the missing stabilization step that produces `stable_models.json`.

In [ ]:
def count_non_zero_entries(up, down):
    """Count matrix entries containing at least one retained insertion."""
    up_counter = 0
    down_counter = 0
    for i in range(3):
        for j in range(3):
            if up[i][j] != []:
                up_counter += 1
            if down[i][j] != []:
                down_counter += 1
    return (up_counter, down_counter)


In [ ]:
try_for_each_model = 50
viable_models = {}
model_counter = 0
for model_id in promising_models:
    if library[model_id][-1] != 1:
        continue
    n_of_non_pert_phi = len(library[model_id][0])
    n_of_phi = n_of_non_pert_phi + len(library[model_id][1])
    for restart in range(try_for_each_model):
        for ckm_attempt in range(CKM_MAX_ATTEMPTS):
            initial_rotations = rng.uniform(*UV_range, size=10)
            result_CKM = minimize(CKM_loss, initial_rotations, method='L-BFGS-B')
            if result_CKM.fun < CKM_threshold:
                Uu = generate_orthogonal_matrix(result_CKM.x[:3], result_CKM.x[3:5])
                Ud = generate_orthogonal_matrix(result_CKM.x[5:8], result_CKM.x[8:10])
                ckm = Uu.T @ Ud
                break
        else:
            print(f'No accepted CKM fit for {model_id}, restart {restart}')
            continue
        for choice_index, sacrificed_vevs in enumerate(killed_vevs_library[model_id]):
            schema, up, down = number_of_fields(matrices[0][model_id], matrices[1][model_id], sacrificed_vevs)
            if min(count_non_zero_entries(up, down)) < 6:
                continue
            n_o1_up, n_o1_down = n_of_o1_coeffs(up, down)
            n_o1 = n_o1_up + 2 * n_o1_down
            n_of_used_Phi = len(schema)
            tot_para = n_of_used_Phi + n_o1 + 13
            bounds = from_symbols_to_bounds(schema, tot_para, True)
            initial_bounds = from_symbols_to_bounds(schema, tot_para, False)
            initial_values = [rng.uniform(low, high) for low, high in initial_bounds]
            solution = least_squares(loss_function, initial_values, args=(up, down, Uu, Ud), method='trf', bounds=([low for low, _ in bounds], [high for _, high in bounds]))
            print(f'{model_id}, restart {restart}, choice {choice_index}: cost={solution.cost:.6g}')
            if solution.cost >= threshold:
                continue
            fitted_matrices = loss_function(solution.x, up, down, Uu, Ud, return_matrices=True)
            masses = [LA.svd(matrix, compute_uv=False) for matrix in fitted_matrices]
            targets = ([Mt, Mc, Mu], [Mb, Ms, Md], [Mtau, Mmu, Me])
            mass_deviations = [[deviation(value, target) for value, target in zip(values, sector_targets)] for values, sector_targets in zip(masses, targets)]
            ckm_deviations = [[deviation(abs(ckm[row, col]), exp_CKM[row][col]) for col in range(3)] for row in range(3)]
            model_counter += 1
            key = f'{model_id}, {restart}:{choice_index}'
            viable_models[key] = [model_counter, float(solution.cost), [up, down], solution.x.tolist(), [matrix.tolist() for matrix in fitted_matrices], [values.tolist() for values in masses], mass_deviations, ckm.tolist(), ckm_deviations, sacrificed_vevs]
with open('viable_models.json', 'w') as file:
    json.dump(viable_models, file, indent=2)
with open('fit_settings.json', 'w') as file:
    json.dump({'seed': RANDOM_SEED, 'restarts': try_for_each_model, 'ckm_max_attempts': CKM_MAX_ATTEMPTS, 'threshold': threshold, 'ckm_threshold': CKM_threshold, 'retained_monomials': n_o1_coef_per_entry, 'nonperturbative_vev_range': non_pert_range, 'perturbative_vev_range': pert_range, 'coefficient_range': o1_range, 'up_higgs_bounds': Hu_bounds, 'up_mass_targets': Mu_h.tolist(), 'down_mass_targets': Md_h.tolist(), 'lepton_mass_targets': Me_h.tolist(), 'ckm_targets': exp_CKM}, file, indent=2)
